In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataX").getOrCreate()

In [0]:
data = {
    ('Abhijeet','IT',50000),
    ('Rohan','HR',60000),
    ('Rohit','Operations',70000),
    ('Pratik','Engineering',80000)
}

column = ['Name','Department','Salary']
df = spark.createDataFrame(data, column)
display(df)

In [0]:
df.write.format("delta").saveAsTable("workspace.default.employee_delta")

In [0]:
%sql
select * from employee_delta

In [0]:
%sql
describe history employee_delta

In [0]:
%sql
update employee_delta set Salary = 65000 where Name = 'Abhijeet'

In [0]:
%sql
describe history employee_delta

In [0]:
#merge INTO - update a row

from pyspark.sql import Row

update_data = [Row(Name = 'Rushikesh', Department = 'Finance', Salary = 70000)]
update_df = spark.createDataFrame(update_data)
update_df.createOrReplaceTempView("update_view")


spark.sql("""
    MERGE INTO employee_delta as TARGET
    USING update_view as SOURCE
    ON TARGET.Name = SOURCE.Name
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")




In [0]:
%sql
DESCRIBE HISTORY employee_delta

In [0]:
%sql
select * from employee_delta

In [0]:
# Schema Evolution (Add new column)

new_data = [('Stuti','Accounts',65000,5)]
cols = ['Name','Department','Salary','Experience']
new_df = spark.createDataFrame(new_data, cols)
display(new_df)

new_df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable("employee_delta")

In [0]:
%sql
SELECT * FROM employee_delta

In [0]:
%sql
DESCRIBE HISTORY employee_delta

In [0]:
%sql
DESCRIBE EXTENDED employee_delta